In [2]:
# ============================================================================
# notebook: notebooks/03_ensemble_and_validity.ipynb  (clean rebuild, new paths)
# Project: "Incidental vs. Engineered Approval"
# Stage 3: build the CONFIRMED 3-axis EngineeredScore + four-fold validity.
#   Ensemble: EngineeredScore = (1/3)*[ Stability_pct + (1 - Density_pct) + NonFragility_pct ]
#   Validity: V2 axis independence | V3 not-confidence | V4 convergent (vs real default)
# Reads results/ ; writes results/. Run from notebooks/.
# ============================================================================


# ─────────────────────────────────────────────────────────────────────────
# CELL 1 — Paths, imports, load Stage-2 indices + Stage-1 cohort
# ─────────────────────────────────────────────────────────────────────────
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm

ROOT    = Path("..").resolve()
DATA    = ROOT / "data"
RESULTS = ROOT / "results"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
AXIS_CORR_THRESHOLD = 0.70
CONF_CORR_MAX = 0.30

idx    = pd.read_parquet(RESULTS / "stage2_indices_approved.parquet")
cohort = pd.read_parquet(RESULTS / "stage1_cohort.parquet")
stage0 = pd.read_parquet(RESULTS / "stage0_labeled.parquet")

AUDIT = (["LIMIT_BAL"] + [f"BILL_AMT{i}" for i in range(1,7)]
         + [f"PAY_AMT{i}" for i in range(1,7)])

idx["DEFAULT"]       = stage0.loc[idx.index, "DEFAULT"].values
idx["P_VIP"]         = cohort.loc[idx.index, "P_VIP_stage1"].values
idx["GROUP"]         = cohort.loc[idx.index, "GROUP"].values
idx["CELL"]          = cohort.loc[idx.index, "CELL"].values
idx["is_borderline"] = cohort.loc[idx.index, "VIP_BORDERLINE_s1"].values
for c in AUDIT:
    idx[c] = stage0.loc[idx.index, c].values

print(f"Approved: {len(idx):,}  |  borderline: {int(idx['is_borderline'].sum()):,}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 2 — Build aligned axes + the CONFIRMED 3-axis EngineeredScore
# ─────────────────────────────────────────────────────────────────────────
idx["A_Stability"]  = idx["Stability_pct"]
idx["A_LowDensity"] = 1.0 - idx["Density_pct"]
idx["A_NonFrag"]    = idx["NonFragility_pct"]
idx["EngineeredScore"] = (idx["A_Stability"] + idx["A_LowDensity"] + idx["A_NonFrag"]) / 3.0

B = idx[idx["is_borderline"] == 1].copy()
print("EngineeredScore (3-axis) built. Borderline analysis set:", len(B),
      f"(defaults: {int(B['DEFAULT'].sum())})")
print(idx["EngineeredScore"].describe().round(3).to_string())


# ─────────────────────────────────────────────────────────────────────────
# CELL 3 — V2: axis independence (the three aligned axes, borderline)
# ─────────────────────────────────────────────────────────────────────────
axes = ["A_Stability", "A_LowDensity", "A_NonFrag"]
corr = B[axes].corr().abs()
max_off = corr.where(~np.eye(3, dtype=bool)).max().max()
print("V2 — aligned-axis absolute correlation (borderline):")
print(corr.round(3).to_string())
print(f"  max off-diagonal |r| = {max_off:.3f}")
V2_PASS = corr.loc["A_Stability", "A_LowDensity"] < AXIS_CORR_THRESHOLD and \
          corr.loc["A_Stability", "A_NonFrag"] < AXIS_CORR_THRESHOLD
print(f"  V2 {'PASS' if V2_PASS else 'REVIEW'} (Stability independent of both)")


# ─────────────────────────────────────────────────────────────────────────
# CELL 4 — V3: not confidence repackaging
# ─────────────────────────────────────────────────────────────────────────
def logit(frame, cols):
    return sm.Logit(frame["DEFAULT"].values, sm.add_constant(frame[cols])).fit(disp=0)

v3a = stats.pearsonr(B["EngineeredScore"], B["P_VIP"])[0]
m_p  = logit(B, ["P_VIP"])
m_ax = logit(B, ["A_Stability", "A_LowDensity", "A_NonFrag", "P_VIP"])
gain = m_ax.prsquared - m_p.prsquared
print(f"V3a — corr(EngineeredScore, p(x)) = {v3a:+.3f}  (< {CONF_CORR_MAX})")
print(f"V3b — pseudo-R2: p(x)-only={m_p.prsquared:.4f}, +axes={m_ax.prsquared:.4f}, gain={gain:+.4f}")
print(f"      p(x) in full model: coef={m_ax.params['P_VIP']:+.3f} (p={m_ax.pvalues['P_VIP']:.2e})")
V3_PASS = (abs(v3a) < CONF_CORR_MAX) and (gain > 0.005)
print(f"  V3 {'PASS' if V3_PASS else 'FAIL'}")


# ─────────────────────────────────────────────────────────────────────────
# CELL 5 — V4: convergent validity (aligned ensemble vs real default, ctrl p(x))
# ─────────────────────────────────────────────────────────────────────────
m_es = logit(B, ["EngineeredScore", "P_VIP"])
coef, pval = m_es.params["EngineeredScore"], m_es.pvalues["EngineeredScore"]
rho, prho = stats.spearmanr(B["EngineeredScore"], B["DEFAULT"])
V4_PASS = (pval < 0.05) and (coef < 0)
print("V4 — convergent validity (borderline, control p(x)):")
print(f"  EngineeredScore coef = {coef:+.3f}  OR={np.exp(coef):.3f}  p={pval:.2e}")
print(f"  Spearman(ES, default) = {rho:+.3f} (p={prho:.2e})")
print(f"  V4 {'PASS' if V4_PASS else 'FAIL'} (correct sign = negative)")


# ─────────────────────────────────────────────────────────────────────────
# CELL 6 — Typicality paradox (density reversal), documented for the paper
# ─────────────────────────────────────────────────────────────────────────
B["dens_q"] = pd.qcut(B["Density_pct"], 4, labels=["Q1_low","Q2","Q3","Q4_high"])
para = B.groupby("dens_q").agg(default_rate=("DEFAULT","mean"), n=("DEFAULT","size")).round(4)
print("Typicality paradox — default by density quartile (borderline):")
print(para.to_string())
B["bill_tot"] = B[[f"BILL_AMT{i}" for i in range(1,7)]].sum(axis=1)
B["pay_tot"]  = B[[f"PAY_AMT{i}" for i in range(1,7)]].sum(axis=1)
m_dens = logit(B, ["Density_pct", "P_VIP", "LIMIT_BAL", "bill_tot", "pay_tot"])
print(f"Density coef vs default (all raw controls): "
      f"{m_dens.params['Density_pct']:+.3f} (p={m_dens.pvalues['Density_pct']:.2e})  "
      f"[+ => dense=risky, paradox]")


# ─────────────────────────────────────────────────────────────────────────
# CELL 7 — RQ2 pre-signal (descriptive; subsampling control is Stage 4)
# ─────────────────────────────────────────────────────────────────────────
grp = B.groupby("GROUP").agg(
    mean_ES=("EngineeredScore","mean"),
    mean_lowdens=("A_LowDensity","mean"),
    mean_stab=("A_Stability","mean"),
    mean_nonfrag=("A_NonFrag","mean"),
    n=("EngineeredScore","size"),
).round(3)
print("RQ2 pre-signal — by group (borderline, DESCRIPTIVE):")
print(grp.to_string())


# ─────────────────────────────────────────────────────────────────────────
# CELL 8 — Persist Stage-3 artifacts
# ─────────────────────────────────────────────────────────────────────────
keep = ["GROUP","CELL","P_VIP","DEFAULT","A_Stability","A_LowDensity","A_NonFrag","EngineeredScore"]
B[keep].to_parquet(RESULTS / "stage3_borderline_scored.parquet")
idx[["GROUP","CELL","P_VIP","DEFAULT","is_borderline",
     "A_Stability","A_LowDensity","A_NonFrag","EngineeredScore"]
    ].to_parquet(RESULTS / "stage3_approved_scored.parquet")
print("Saved Stage-3 scored artifacts to results/.")


# ─────────────────────────────────────────────────────────────────────────
# CELL 9 — Stage 3 summary
# ─────────────────────────────────────────────────────────────────────────
print("=" * 68)
print("STAGE 3 — 3-AXIS ENSEMBLE + FOUR-FOLD VALIDITY")
print("=" * 68)
print(f"Ensemble  : (1/3)*(Stability + LowDensity + NonFragility)")
print(f"V2 indep. : Stability vs others {corr.loc['A_Stability','A_LowDensity']:.2f}/"
      f"{corr.loc['A_Stability','A_NonFrag']:.2f}  {'PASS' if V2_PASS else 'REVIEW'}")
print(f"V3 not-conf: corr_p(x)={v3a:+.3f}, gain={gain:+.4f}  {'PASS' if V3_PASS else 'FAIL'}")
print(f"V4 converg.: coef={coef:+.3f} (p={pval:.2e})  {'PASS' if V4_PASS else 'FAIL'}")
print(f"Paradox   : density coef (all ctrl) = {m_dens.params['Density_pct']:+.3f}")
print("-" * 68)
print("OVERALL:", "CONFIRMED — proceed to Stage 4" if (V2_PASS and V3_PASS and V4_PASS)
      else "REVIEW a failed check")
print("=" * 68)

Approved: 11,089  |  borderline: 1,141
EngineeredScore (3-axis) built. Borderline analysis set: 1141 (defaults: 144)
count    11089.000
mean         0.500
std          0.195
min          0.010
25%          0.353
50%          0.491
75%          0.650
max          0.991
V2 — aligned-axis absolute correlation (borderline):
              A_Stability  A_LowDensity  A_NonFrag
A_Stability          1.00         0.270      0.330
A_LowDensity         0.27         1.000      0.695
A_NonFrag            0.33         0.695      1.000
  max off-diagonal |r| = 0.695
  V2 PASS (Stability independent of both)
V3a — corr(EngineeredScore, p(x)) = +0.002  (< 0.3)
V3b — pseudo-R2: p(x)-only=0.0012, +axes=0.0159, gain=+0.0146
      p(x) in full model: coef=+0.884 (p=5.78e-01)
  V3 PASS
V4 — convergent validity (borderline, control p(x)):
  EngineeredScore coef = -1.249  OR=0.287  p=3.98e-02
  Spearman(ES, default) = -0.074 (p=1.23e-02)
  V4 PASS (correct sign = negative)
Typicality paradox — default by densi